In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.model as _model_mod
importlib.reload(_model_mod)
from src.model import saint_venant_1d
from src import sensitivity as sab

sinteticos = sab.load_sinteticos_module(ROOT)

PARAM_NAMES = ["n", "S0", "B_W", "eta_Q"]
PARAMS_TRUE_N_S0 = [sinteticos.N_MANN, sinteticos.S0]
L_CANAL = sinteticos.L
NX = 100
WARMUP_SECONDS = 3600.0
BOUNDS_LO_N_S0 = np.array([0.01, 0.0001])
BOUNDS_HI_N_S0 = np.array([0.06, 0.005])
BW_REF = sinteticos.B_W
ETA_K_SIGMA = 2.0
BW_REL = 0.4
WEIGHT_UP, WEIGHT_DN = 0.5, 0.5
CAL_FRAC, VAL_FRAC = 0.60, 0.30

FIG = ROOT / "figures"
DATA = ROOT / "data" / "synthetic"
for d in (FIG, DATA):
    d.mkdir(parents=True, exist_ok=True)

FAST = True
MC_N = 200 if FAST else 50_000
GLUE_KGE_MIN = 0.5
import os
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
USE_NOISY_CSV = False
RNG_SEED = 42

print("ROOT:", ROOT, "| FAST:", FAST, "| MC_N:", MC_N)


In [ ]:
if USE_NOISY_CSV:
    csv_name = "series_corta_ruido.csv"
else:
    csv_name = "series_corta_balance.csv"

csv_path = DATA / csv_name
if not csv_path.exists():
    print("Generando CSV con data/sinteticos.py ...")
    sinteticos.generate(str(DATA))

df = pd.read_csv(csv_path, parse_dates=["datetime"])
t_sec = (df["datetime"] - df["datetime"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)
q_up = df["Q_upstream_m3s"].to_numpy(dtype=float)
q_obs = df["Q_downstream_m3s"].to_numpy(dtype=float)
nt = len(df)

mask_warm = t_sec < WARMUP_SECONDS
idx_post = np.where(~mask_warm)[0]
n_post = len(idx_post)
n_cal = max(1, int(CAL_FRAC * n_post))
n_val = max(1, int(VAL_FRAC * n_post))
n_cal = min(n_cal, n_post - n_val)
i_cal = idx_post[:n_cal]
i_val = idx_post[n_cal : n_cal + n_val]

mask_cal = np.zeros(nt, dtype=bool)
mask_val = np.zeros(nt, dtype=bool)
mask_cal[i_cal] = True
mask_val[i_val] = True

SIM = sab.TwoStationSimulator(
    saint_venant_1d, q_up, t_sec, nt, canal_L=L_CANAL, nx=NX
)

_, eta_lo, eta_hi = sab._eta_q_bounds(q_up, mask_cal, k_sigma=ETA_K_SIGMA)
_, bw_lo, bw_hi = sab._bw_bounds(BW_REF, rel=BW_REL)
PARAMS_TRUE = PARAMS_TRUE_N_S0 + [BW_REF, 1.0]
BOUNDS_LO = np.concatenate([BOUNDS_LO_N_S0, [bw_lo, eta_lo]])
BOUNDS_HI = np.concatenate([BOUNDS_HI_N_S0, [bw_hi, eta_hi]])
print("CSV:", csv_name, " cal=", mask_cal.sum(), " val=", mask_val.sum())
print("Vector MC/GLUE:", PARAM_NAMES)


In [ ]:
def kge_two(params):
    qu, qd = SIM.both(params)
    ku = sab.kge(q_up, qu, mask_cal)
    kd = sab.kge(q_obs, qd, mask_cal)
    return WEIGHT_UP * ku + WEIGHT_DN * kd


def simulate_outlet(params):
    """Salida en x=L (compatibilidad con figuras GLUE)."""
    return SIM.both(params)[1]


def eval_kge_cal(params):
    return kge_two(params)


## Monte Carlo exploratorio

No filtra por desempeño: solo muestra la dispersión de KGE al sortear θ uniforme en los rangos.


In [ ]:
rng = np.random.default_rng(RNG_SEED)
samples_mc = rng.uniform(BOUNDS_LO, BOUNDS_HI, size=(MC_N, len(PARAM_NAMES)))

print("Monte Carlo:", MC_N, "simulaciones...")
kge_mc = np.array(
    Parallel(n_jobs=N_JOBS)(delayed(eval_kge_cal)(samples_mc[i]) for i in range(MC_N))
)

pd.DataFrame(
    {
        "n": samples_mc[:, 0],
        "S0": samples_mc[:, 1],
        "B_W": samples_mc[:, 2],
        "eta_Q": samples_mc[:, 3],
        "KGE_cal": kge_mc,
    }
).to_csv(DATA / "monte_carlo_resumen.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
finite = kge_mc[np.isfinite(kge_mc)]
ax.hist(finite, bins=min(30, max(5, MC_N // 10)), edgecolor="k", alpha=0.75)
ax.axvline(GLUE_KGE_MIN, color="r", ls="--", label=f"umbral GLUE KGE>={GLUE_KGE_MIN}")
ax.set_xlabel("KGE (calibracion)")
ax.set_ylabel("frecuencia")
ax.set_title(f"Monte Carlo ({MC_N} sorteos uniformes)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "monte_carlo_kge_hist.png", dpi=150)
plt.show()
print("Guardado:", DATA / "monte_carlo_resumen.csv")


## Diagramas de dispersion (referencia `MonteCarlo - Copy.ipynb`)

Tras el barrido Monte Carlo: **parametro vs desempeño** (1 − KGE en cal), version global y solo conjuntos **conductuales** (mismo criterio GLUE).


In [ ]:
from src.calibration_plots import plot_obs_vs_sim, plot_param_vs_objective

of_mc = 1.0 - np.where(np.isfinite(kge_mc), kge_mc, np.nan)
thr_of = 1.0 - GLUE_KGE_MIN  # equivalente a KGE >= umbral

plot_param_vs_objective(
    samples_mc,
    of_mc,
    PARAM_NAMES,
    y_label="1 - KGE (cal)",
    title="Monte Carlo — exploracion del espacio de parametros",
    save_path=FIG / "scatter_param_vs_objetivo_todas.png",
)
plt.show()

plot_param_vs_objective(
    samples_mc,
    of_mc,
    PARAM_NAMES,
    y_label="1 - KGE (cal)",
    title="Monte Carlo — conjuntos conductuales",
    threshold=thr_of,
    save_path=FIG / "scatter_param_vs_objetivo_conductuales.png",
)
plt.show()

i_best = int(np.nanargmax(kge_mc))
q_best = simulate_outlet(samples_mc[i_best])
plot_obs_vs_sim(
    q_obs,
    q_best,
    mask=mask_cal,
    title=f"Mejor MC (KGE cal={kge_mc[i_best]:.3f})",
    save_path=FIG / "scatter_Qsim_vs_Qobs_mejor_MC.png",
)
plt.show()
print("Guardadas: scatter_param_vs_objetivo_*.png, scatter_Qsim_vs_Qobs_mejor_MC.png")


## GLUE — bandas de credibilidad 5–95 %

Conjuntos **conductuales**: simulaciones con `KGE_cal >= GLUE_KGE_MIN`.
La envolvente 5–95 % se calcula en cada instante (figura post warm-up).


In [ ]:
from src.calibration_plots import glue_bands, plot_hydrograph_glue

behavioral = samples_mc[kge_mc >= GLUE_KGE_MIN]
n_beh = len(behavioral)
print(f"GLUE: {n_beh} / {MC_N} conjuntos conductuales (KGE >= {GLUE_KGE_MIN})")

if n_beh == 0:
    raise RuntimeError(
        "Ningun set conductual. Baje GLUE_KGE_MIN o aumente MC_N (FAST=False)."
    )

print("Simulando hidrogramas conductuales...")
Qs = np.array(
    Parallel(n_jobs=N_JOBS)(delayed(simulate_outlet)(behavioral[i]) for i in range(n_beh))
)

p05, p50, p95 = glue_bands(Qs, alpha=0.05)
i_best_glue = int(np.argmax([kge(q_obs, Qs[i], mask_cal) for i in range(n_beh)]))
q_best_glue = Qs[i_best_glue]

pd.DataFrame(
    {
        "n": behavioral[:, 0],
        "S0": behavioral[:, 1],
        "B_W": behavioral[:, 2],
        "eta_Q": behavioral[:, 3],
    }
).to_csv(DATA / "glue_parametros_aceptados.csv", index=False)

pd.DataFrame(
    {
        "datetime": df["datetime"],
        "Q_obs_m3s": q_obs,
        "Q_p05_m3s": p05,
        "Q_p50_m3s": p50,
        "Q_p95_m3s": p95,
    }
).to_csv(DATA / "glue_envelope_xL.csv", index=False)

t_h = t_sec / 3600.0
plot_hydrograph_glue(
    t_h,
    q_obs,
    Qs,
    q_best_glue,
    mask_plot=~mask_warm,
    warmup_hours=WARMUP_SECONDS / 3600.0,
    alpha=0.05,
    title=f"GLUE — {n_beh} sets conductuales (KGE cal >= {GLUE_KGE_MIN})",
    save_path=FIG / "glue_credibilidad_xL.png",
)
plt.show()

m_plot = ~mask_warm
inside = (q_obs[m_plot] >= p05[m_plot]) & (q_obs[m_plot] <= p95[m_plot])
print("Observaciones dentro de la banda (post warm-up):", f"{inside.mean():.1%}")
print("Guardado:", DATA / "glue_envelope_xL.csv")
